In [0]:
from main import UC_TOOL_NAMES,VECTOR_SEARCH_TOOL

import mlflow
from mlflow.models.resources import DatabricksFunction
from pkg_resources import get_distribution


resources = []

resources.extend(VECTOR_SEARCH_TOOL.resources)


for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))


with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="foodly-ai-assistant",
        code_paths=["helpers.py"],
        python_model='main.py',
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
             f"langgraph-supervisor=={get_distribution('langgraph-supervisor').version}",
        ],
        resources=resources,
    )

🔗 View Logged Model at: https://8259551295636485.5.gcp.databricks.com/ml/experiments/16a101a6d89648ecbd38b4d419a761c3/models/m-f02b87f0411649cd91ee96ff20ef81cc?o=8259551295636485
2026/06/22 10:48:42 INFO mlflow.pyfunc: Predicting on input example to validate output

### Pre-deployment agent validation


In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/foodly-ai-assistant",
    input_data={"input": [{"role": "user", "content": "Tell me about foodly return policy"}]},
    env_manager="uv",
)

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "agents"
schema = "main"
model_name = "foodly-ai-assistant"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

Successfully registered model 'agents.main.foodly-ai-assistant'.

In [0]:
from databricks import agents

agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    scale_to_zero=True,
    tags={"name": "foodly-ai-assistant-endpoint","creator" : "anirvan-sen"},
)


    Deployment of agents.main.foodly-ai-assistant version 1 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://8259551295636485.5.gcp.databricks.com/ml/endpoints/agents_agents-main-foodly-ai-assistant
    Review App: https://8259551295636485.5.gcp.databricks.com/ml/review-v2/5bc6093206f047f5b2faf873d99b25c0/chat
    Monitor: https://8259551295636485.5.gcp.databricks.com/ml/experiments/16a101a6d89648ecbd38b4d419a761c3?compareRunsMode=TRACES
Deployment(model_name='agents.main.foodly-ai-assistant', model_version='1', endpoint_name='agents_agents-main-foodly-ai-assistant', served_entity_name='agents-main-foodly-ai-assistant_1', query_endpoint='https://8259551295636485.5.gcp.databricks.com/serving-endpoints/agents_agents-main-foodly-ai-assistant/served-models/agents-main-foodly-ai-assistant_1/invocations', endpoint_url='https://8259551295636485.5.gcp.databricks.com/ml/endpoints/agents_agents-main-foodly-ai-assistant', review_app_url='https://8259551295636485.5.gcp.databricks.com/ml/review-v2/5bc6093206f047f5b2faf873d99b25c0/chat')

[](url)

[](url)

[](url)

Tool call: foodly_policy_document_retrieval_tool
{
  "query": "refund policy",
  "filters": null
}

Tool call output:
[Document(metadata={'chunk_id': 1.0}, page_content='FOODLY - CUSTOMER SERVICE & OPERATIONS POLICY (Version 1.0)\n\n1. INTRODUCTION\nWelcome to Foodly, your trusted food delivery partner.\nThis document serves as a unified reference for all customer-related policies, including:\n- Refunds & Cancellations\n- Order & Delivery Issues\n- Loyalty & Rewards\n- Privacy & Data Protection\n- Fraud Prevention & Escalation\nBy using Foodly’s platform, you accept the terms below.\n\n2. REFUND & COMPENSATION POLICY\n\n2.1 Refund Eligibility\nRefunds are offered when:\n- ORDER NOT DELIVERED:\n  Causes: restaurant closure, unavailable delivery partners, extreme weather, or technical failures.\n  Resolution: Full refund to original payment method or Foodly credits.\n- INCORRECT OR MISSING ITEMS:\n  Examples: wrong dish, missing beverages, incorrect quantity.\n  Resolution: Partial refund or credits equivalent to missing items.\n- FOOD QUALITY ISSUES:\n  Spoiled, unsafe, or contaminated food.\n  Resolution: Full refund or re-delivery at no extra cost (where available).\n- DUPLICATE PAYMENT:\n  Accidental multiple charges.\n  Resolution: Refund of the duplicate transaction(s).\n\n2.2 Non-Refundable Cases\nRefunds will NOT be issued for:\n- Incorrect address provided by customer.\n- Customer unreachable at time of delivery.\n- Taste dissatisfaction (e.g., too spicy, too bland).\n- Delay caused by external events (traffic, weather, local restrictions) unless delivery exceeds 90 minutes beyond ETA.\n\n2.3 Refund Timelines\n- Bank or card refunds: 5–7 business days.\n- Foodly credits: Instant after approval.\n- Confirmation sent via in-app notification and email.\n\n3. ORDER CANCELLATIONS\n- CUSTOMER-INITIATED:\n  Within 2 minutes of order placement: free cancellation.\n  After 2 minutes: subject to restaurant acceptance; partial or no refund if preparation started.\n- RESTAURANT OR FOODLY-INITIATED:\n  Due to unavailability, operational constraints, or safety concerns.\n  Full refund or credits will be issued.\n\n4. DELIVERY OPERATIONS\n- ESTIMATED DELIVERY TIME (ETA):\n  Displayed at checkout; may vary based on traffic, weather, and restaurant preparation times.\n- LATE DELIVERY COMPENSATION:\n  If delivery exceeds ETA by 30 minutes or more:\n  Foodly credits worth 10% of order value (max ₹100).\n- RE-DELIVERY:\n  Offered in cases of incorrect or incomplete orders where feasible.\n\n5. LOYALTY & REWARDS PROGRAM\n- EARNING POINTS:\n  Earn 1 point per ₹50 spent.\n  Bonus points for using select payment methods or during promotional periods.\n- REDEEMING POINTS:\n  Every 100 points = ₹50 discount on next order.\n  Cannot be exchanged for cash.\n- SPECIAL TIERS:\n  Gold Tier: Unlocks after ₹10,000 total spend; benefits include priority support and free delivery on select orders.\n\n6. PRIVACY & DATA PROTECTION\n- We collect minimal personal data: name, contact details, delivery address, and payment information.\n- Data is encrypted and never sold to third parties.\n- Customers can request account deletion and data removal by contacting support.\n\n7. FRAUD PREVENTION & ABUSE CONTROL\n- Multiple suspicious refund claims without evidence may trigger account review.\n- Confirmed fraudulent activity may result in:\n  Temporary suspension.\n  Permanent account closure.\n  Legal escalation where applicable.\n\n8. ESCALATION & SUPPORT\n- FIRST LEVEL: In-app chat or call via “Help” section.\n- SECOND LEVEL: Email escalation to support@foodly.com.\n- RESPONSE SLA: Within 24 hours for non-urgent, 4 hours for urgent cases.\n\n9. POLICY UPDATES\n- Foodly reserves the right to amend these policies.\n- All updates will be posted in-app and on the website.\n- Continued use of Foodly after updates implies acceptance.\n')]

Based on the provided policy document, here is a detailed overview of the refund policy in a tabular format:

### Refund Eligibility

| Reason for Refund | Resolution |
| --- | --- |
| Order Not Delivered | Full refund to original payment method or Foodly credits |
| Incorrect or Missing Items | Partial refund or credits equivalent to missing items |
| Food Quality Issues | Full refund or re-delivery at no extra cost (where available) |
| Duplicate Payment | Refund of the duplicate transaction(s) |

### Non-Refundable Cases

| Reason | Explanation |
| --- | --- |
| Incorrect Address | Provided by the customer |
| Customer Unreachable | At the time of delivery |
| Taste Dissatisfaction | Too spicy, too bland, etc. |
| Delay Due to External Events | Unless delivery exceeds 90 minutes beyond ETA |

### Refund Timelines

| Refund Method | Timeline |
| --- | --- |
| Bank or Card Refunds | 5–7 business days |
| Foodly Credits | Instant after approval |

### Order Cancellations

| Type of Cancellation | Terms |
| --- | --- |
| Customer-Initiated (within 2 minutes) | Free cancellation |
| Customer-Initiated (after 2 minutes) | Subject to restaurant acceptance; partial or no refund if preparation started |
| Restaurant or Foodly-Initiated | Full refund or credits will be issued due to unavailability, operational constraints, or safety concerns |

For more details or specific inquiries regarding the refund policy or any other Foodly policies, you can reach out to their support team through the in-app chat, call, or email escalation to support@foodly.com.